# 07 - Model: KNN (K-Nearest Neighbors)


# Imports and Load Data

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv('../data/processed/featured_dataset.csv')
df_full_standard = pd.read_csv('../data/processed/featured_full_standard.csv')
df_full_minmax = pd.read_csv('../data/processed/featured_full_minmax.csv')
df_audio_standard = pd.read_csv('../data/processed/featured_audio_standard.csv')
df_audio_minmax = pd.read_csv('../data/processed/featured_audio_minmax.csv')

with open('../config/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

audio_features = feature_sets['audio_features']
full_features = feature_sets['full_features']

print('shape:', df.shape)
print('audio features:', len(audio_features))
print('full features:', len(full_features))

shape: (88167, 32)
audio features: 11
full features: 21


# KNN Recommendation Function

In [2]:
def recommend_knn(playlist_songs, df, features, scaled_df, k=10, n=10):
    # find songs in playlist
    playlist_indices = []
    not_found = []
    
    for song in playlist_songs:
        matches = df[df['track_name'].str.lower() == song.lower()]
        if not matches.empty:
            playlist_indices.append(matches.index[0])
        else:
            not_found.append(song)
    
    if not_found:
        print(f'Songs not found: {not_found}')
    
    if not playlist_indices:
        print('No songs found in playlist')
        return None
    
    print(f'Playlist songs found: {len(playlist_indices)}/{len(playlist_songs)}')
    print('Playlist:')
    for idx in playlist_indices:
        print(f'  - {df.loc[idx, "track_name"]} by {df.loc[idx, "artists"]}')
    print()
    
    # compute playlist average features
    playlist_features = scaled_df[features].iloc[playlist_indices].mean().values.reshape(1, -1)
    
    # fit KNN
    knn = NearestNeighbors(n_neighbors=k+len(playlist_indices), metric='euclidean')
    knn.fit(scaled_df[features].values)
    
    # find nearest neighbors
    distances, indices = knn.kneighbors(playlist_features)
    
    # exclude playlist songs
    recommended_indices = [i for i in indices[0] if i not in playlist_indices][:n]
    
    results = df.iloc[recommended_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['distance'] = distances[0][:len(recommended_indices)]
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

# Experiment 1 - Full Features StandardScaler K=10

In [3]:
playlist = [
    'Shape of You',
    'Blinding Lights',
    'Dance Monkey',
    'Watermelon Sugar',
    'Levitating'
]

print('K=10 | Features: full | Scaling: StandardScaler\n')
results = recommend_knn(playlist, df, full_features, df_full_standard, k=10)
print(results)

K=10 | Features: full | Scaling: StandardScaler

Playlist songs found: 5/5
Playlist:
  - Shape Of You by Andrew Foy
  - Blinding Lights by Kidz Bop Kids
  - Dance Monkey by Roses & Frey
  - Watermelon Sugar by Billboard Baby Lullabies
  - Levitating by Kidz Bop Kids

              track_name              artists track_genre  popularity  \
1         Call Me a Fool                Perlo     ambient          40   
2        All Things Pass      Federico Aubele    trip-hop          18   
3                 Dígale        Chamin Correa      guitar          23   
4            I'm On Fire           Chromatics   synth-pop          50   
5          Harry Houdini  Tomas Andersson Wij     swedish          35   
6                    Two               Seekae         idm           8   
7             Impossible             Lyla Foy     british          43   
8            Ты моё море               Kvatro     romance           7   
9        Bas gillar hörn      Veronica Maggio     swedish          37   
10

# Improved KNN Function with Genre Filter

In [4]:
def recommend_knn_v2(playlist_songs, df, features, scaled_df, k=10, n=10):
    # find most popular version of each song
    playlist_indices = []
    not_found = []
    
    for song in playlist_songs:
        matches = df[df['track_name'].str.lower() == song.lower()]
        if not matches.empty:
            # take most popular version
            best_idx = matches['popularity'].idxmax()
            playlist_indices.append(best_idx)
        else:
            not_found.append(song)
    
    if not_found:
        print(f'Songs not found: {not_found}')
    
    if not playlist_indices:
        print('No songs found in playlist')
        return None
    
    print(f'Playlist songs found: {len(playlist_indices)}/{len(playlist_songs)}')
    print('Playlist:')
    for idx in playlist_indices:
        print(f'  - {df.loc[idx, "track_name"]} by {df.loc[idx, "artists"]} (popularity: {df.loc[idx, "popularity"]})')
    print()
    
    # get dominant genre from playlist
    playlist_genres = []
    for idx in playlist_indices:
        genres = df.loc[idx, 'track_genre'].split(', ')
        playlist_genres.extend(genres)
    
    # find most common genre
    from collections import Counter
    dominant_genre = Counter(playlist_genres).most_common(1)[0][0]
    print(f'Dominant genre: {dominant_genre}\n')
    
    # filter by dominant genre
    genre_mask = df['track_genre'].str.contains(dominant_genre)
    filtered_df = df[genre_mask]
    filtered_scaled = scaled_df[genre_mask]
    
    # compute playlist average
    playlist_features = scaled_df[features].iloc[playlist_indices].mean().values.reshape(1, -1)
    
    # fit KNN
    knn = NearestNeighbors(n_neighbors=k+len(playlist_indices), metric='euclidean')
    knn.fit(filtered_scaled[features].values)
    
    distances, indices = knn.kneighbors(playlist_features)
    
    # exclude playlist songs
    actual_indices = filtered_df.index[indices[0]]
    recommended = [(idx, dist) for idx, dist in zip(actual_indices, distances[0]) 
                   if idx not in playlist_indices][:n]
    
    rec_indices = [r[0] for r in recommended]
    rec_distances = [r[1] for r in recommended]
    
    results = df.loc[rec_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['distance'] = rec_distances
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

# test
print('K=10 | Features: full | Scaling: StandardScaler | Genre filtered\n')
results = recommend_knn_v2(playlist, df, full_features, df_full_standard, k=10)
print(results)

K=10 | Features: full | Scaling: StandardScaler | Genre filtered

Playlist songs found: 5/5
Playlist:
  - Shape of You by Ed Sheeran (popularity: 86)
  - Blinding Lights by The Weeknd (popularity: 91)
  - Dance Monkey by Refeci;Michel Fannoun (popularity: 53)
  - Watermelon Sugar by Harry Styles (popularity: 89)
  - Levitating by Dua Lipa (popularity: 84)

Dominant genre: pop

                           track_name          artists track_genre  \
1           Cigarettes out the Window          TV Girl   indie-pop   
2                          Devil Town         Cavetown   indie-pop   
3                                  시작             Gaho       k-pop   
4                          Start Over             Gaho       k-pop   
5   Everybody Wants To Rule The World  Tears For Fears   synth-pop   
6                      Ready For Love        BLACKPINK       k-pop   
7                         Blank Space     Taylor Swift         pop   
8     Good Times Roll - 2016 Remaster         The Cars   pow

# Experiment 2 - Different K Values

In [5]:
for k in [5, 10, 20, 50]:
    print(f'\n{"="*40}')
    print(f'K={k} | Features: full | Scaling: StandardScaler')
    print('='*40)
    results = recommend_knn_v2(playlist, df, full_features, df_full_standard, k=k)
    if results is not None:
        print(results)


K=5 | Features: full | Scaling: StandardScaler
Playlist songs found: 5/5
Playlist:
  - Shape of You by Ed Sheeran (popularity: 86)
  - Blinding Lights by The Weeknd (popularity: 91)
  - Dance Monkey by Refeci;Michel Fannoun (popularity: 53)
  - Watermelon Sugar by Harry Styles (popularity: 89)
  - Levitating by Dua Lipa (popularity: 84)

Dominant genre: pop

                           track_name          artists track_genre  \
1           Cigarettes out the Window          TV Girl   indie-pop   
2                          Devil Town         Cavetown   indie-pop   
3                                  시작             Gaho       k-pop   
4                          Start Over             Gaho       k-pop   
5   Everybody Wants To Rule The World  Tears For Fears   synth-pop   
6                      Ready For Love        BLACKPINK       k-pop   
7                         Blank Space     Taylor Swift         pop   
8     Good Times Roll - 2016 Remaster         The Cars   power-pop   
9       

# Experiment 3 - Different Distance Metrics

In [6]:
for metric in ['euclidean', 'manhattan', 'cosine']:
    print(f'\n{"="*40}')
    print(f'Metric: {metric} | K=10 | full features | StandardScaler')
    print('='*40)
    
    # modify function to accept metric
    playlist_indices = []
    for song in playlist:
        matches = df[df['track_name'].str.lower() == song.lower()]
        if not matches.empty:
            playlist_indices.append(matches['popularity'].idxmax())
    
    dominant_genre = 'pop'
    genre_mask = df['track_genre'].str.contains(dominant_genre)
    filtered_df = df[genre_mask]
    filtered_scaled = df_full_standard[genre_mask]
    
    playlist_features = df_full_standard[full_features].iloc[playlist_indices].mean().values.reshape(1, -1)
    
    knn = NearestNeighbors(n_neighbors=20, metric=metric)
    knn.fit(filtered_scaled[full_features].values)
    
    distances, indices = knn.kneighbors(playlist_features)
    actual_indices = filtered_df.index[indices[0]]
    recommended = [(idx, dist) for idx, dist in zip(actual_indices, distances[0])
                   if idx not in playlist_indices][:10]
    
    rec_indices = [r[0] for r in recommended]
    rec_distances = [r[1] for r in recommended]
    
    results = df.loc[rec_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['distance'] = rec_distances
    results = results.reset_index(drop=True)
    results.index += 1
    print(results)


Metric: euclidean | K=10 | full features | StandardScaler
                           track_name          artists track_genre  \
1           Cigarettes out the Window          TV Girl   indie-pop   
2                          Devil Town         Cavetown   indie-pop   
3                                  시작             Gaho       k-pop   
4                          Start Over             Gaho       k-pop   
5   Everybody Wants To Rule The World  Tears For Fears   synth-pop   
6                      Ready For Love        BLACKPINK       k-pop   
7                         Blank Space     Taylor Swift         pop   
8     Good Times Roll - 2016 Remaster         The Cars   power-pop   
9                              もう少しだけ          YOASOBI       j-pop   
10                    Good Times Roll         The Cars   power-pop   

    popularity  distance  
1           80  1.432352  
2           78  1.573655  
3           65  1.662435  
4           60  1.662435  
5           87  1.667318  
6       

# Experiment 4 - Audio Features vs Full Features (Cosine Metric)

In [7]:
for features, scaled_df, label in [
    (audio_features, df_audio_standard, 'audio + standard'),
    (full_features, df_full_standard, 'full + standard'),
    (full_features, df_full_minmax, 'full + minmax'),
]:
    print(f'\n{"="*40}')
    print(f'Features: {label}')
    print('='*40)

    playlist_indices = []
    for song in playlist:
        matches = df[df['track_name'].str.lower() == song.lower()]
        if not matches.empty:
            playlist_indices.append(matches['popularity'].idxmax())

    genre_mask = df['track_genre'].str.contains('pop')
    filtered_df = df[genre_mask]
    filtered_scaled = scaled_df[genre_mask]

    playlist_features = scaled_df[features].iloc[playlist_indices].mean().values.reshape(1, -1)

    knn = NearestNeighbors(n_neighbors=20, metric='cosine')
    knn.fit(filtered_scaled[features].values)

    distances, indices = knn.kneighbors(playlist_features)
    actual_indices = filtered_df.index[indices[0]]
    recommended = [(idx, dist) for idx, dist in zip(actual_indices, distances[0])
                   if idx not in playlist_indices][:10]

    rec_indices = [r[0] for r in recommended]
    rec_distances = [r[1] for r in recommended]

    results = df.loc[rec_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['distance'] = rec_distances
    results = results.reset_index(drop=True)
    results.index += 1
    print(results)


Features: audio + standard
                               track_name  \
1   Kusu Kusu (From "Satyameva Jayate 2")   
2                       21st Century Girl   
3                                Who Says   
4                               PING PONG   
5                    Dance The Night Away   
6                                 Charmer   
7         Further Up (Na, Na, Na, Na, Na)   
8                       Алкоголь мой враг   
9                       Алкоголь мой враг   
10                              I Love It   

                                  artists track_genre  popularity  distance  
1   Tanishk Bagchi;Zahrah S Khan;Dev Negi         pop          71  0.110090  
2                                     BTS       k-pop          65  0.114476  
3                Selena Gomez & The Scene  dance, pop          77  0.126972  
4                              HyunA&DAWN       k-pop          68  0.129599  
5                                   TWICE       k-pop          69  0.136004  
6       

# Improved KNN - Weighted Average + Multiple Genre

In [10]:
def recommend_knn_v3(playlist_songs, df, features, scaled_df, n=10):
    # find most popular version of each song
    playlist_indices = []
    not_found = []
    
    for song in playlist_songs:
        matches = df[df['track_name'].str.lower() == song.lower()]
        if not matches.empty:
            best_idx = matches['popularity'].idxmax()
            playlist_indices.append(best_idx)
        else:
            not_found.append(song)
    
    if not_found:
        print(f'Songs not found: {not_found}')
    
    if not playlist_indices:
        print('No songs found in playlist')
        return None
    
    print(f'Playlist songs found: {len(playlist_indices)}/{len(playlist_songs)}')
    print('Playlist:')
    for idx in playlist_indices:
        print(f'  - {df.loc[idx, "track_name"]} by {df.loc[idx, "artists"]} (popularity: {df.loc[idx, "popularity"]})')
    print()
    
    # weighted average by popularity
    popularities = df.loc[playlist_indices, 'popularity'].values
    weights = popularities / popularities.sum()
    
    playlist_features = np.average(
        scaled_df[features].iloc[playlist_indices].values,
        axis=0,
        weights=weights
    ).reshape(1, -1)
    
    # collect all genres from playlist
    all_genres = set()
    for idx in playlist_indices:
        genres = df.loc[idx, 'track_genre'].split(', ')
        all_genres.update(genres)
    
    print(f'Playlist genres: {all_genres}\n')
    
    # filter by any matching genre
    def has_common_genre(genre_str):
        song_genres = set(genre_str.split(', '))
        return bool(song_genres & all_genres)
    
    genre_mask = df['track_genre'].apply(has_common_genre)
    filtered_df = df[genre_mask]
    filtered_scaled = scaled_df[genre_mask]
    
    print(f'Songs in genre pool: {len(filtered_df)}')
    
    # fit KNN with cosine metric
    knn = NearestNeighbors(n_neighbors=20, metric='cosine')
    knn.fit(filtered_scaled[features].values)
    
    distances, indices = knn.kneighbors(playlist_features)
    actual_indices = filtered_df.index[indices[0]]
    recommended = [(idx, dist) for idx, dist in zip(actual_indices, distances[0])
                   if idx not in playlist_indices][:n]
    
    rec_indices = [r[0] for r in recommended]
    rec_distances = [r[1] for r in recommended]
    
    results = df.loc[rec_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['distance'] = rec_distances
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

# test
print('KNN v3 - Weighted Average + Multiple Genre\n')
results = recommend_knn_v3(playlist, df, full_features, df_full_standard)
print(results)

KNN v3 - Weighted Average + Multiple Genre

Playlist songs found: 5/5
Playlist:
  - Shape of You by Ed Sheeran (popularity: 86)
  - Blinding Lights by The Weeknd (popularity: 91)
  - Dance Monkey by Refeci;Michel Fannoun (popularity: 53)
  - Watermelon Sugar by Harry Styles (popularity: 89)
  - Levitating by Dua Lipa (popularity: 84)

Playlist genres: {'pop', 'electronic', 'dance', 'deep-house'}

Songs in genre pool: 3727
                  track_name                   artists track_genre  \
1           Take You Dancing              Jason Derulo       dance   
2           Sweet but Psycho                   Ava Max       dance   
3   What Makes You Beautiful             One Direction         pop   
4                   Who Says  Selena Gomez & The Scene  dance, pop   
5                   Dynamite                       BTS  k-pop, pop   
6            Don't Start Now                  Dua Lipa       dance   
7                  New Rules                  Dua Lipa       dance   
8            D

In [11]:
def recommend_knn_v4(playlist_songs, df, features, scaled_df, n=10):
    playlist_indices = []
    not_found = []
    
    for song in playlist_songs:
        matches = df[df['track_name'].str.lower() == song.lower()]
        if not matches.empty:
            best_idx = matches['popularity'].idxmax()
            playlist_indices.append(best_idx)
        else:
            not_found.append(song)
    
    if not_found:
        print(f'Songs not found: {not_found}')
    
    if not playlist_indices:
        return None
    
    print(f'Playlist songs found: {len(playlist_indices)}/{len(playlist_songs)}')
    print('Playlist:')
    for idx in playlist_indices:
        print(f'  - {df.loc[idx, "track_name"]} by {df.loc[idx, "artists"]} (popularity: {df.loc[idx, "popularity"]})')
    print()
    
    # weighted average by popularity
    popularities = df.loc[playlist_indices, 'popularity'].values
    weights = popularities / popularities.sum()
    playlist_features = np.average(
        scaled_df[features].iloc[playlist_indices].values,
        axis=0, weights=weights).reshape(1, -1)
    
    # collect all genres
    all_genres = set()
    for idx in playlist_indices:
        genres = df.loc[idx, 'track_genre'].split(', ')
        all_genres.update(genres)
    print(f'Playlist genres: {all_genres}\n')
    
    # filter by any matching genre
    def has_common_genre(genre_str):
        song_genres = set(genre_str.split(', '))
        return bool(song_genres & all_genres)
    
    genre_mask = df['track_genre'].apply(has_common_genre)
    filtered_df = df[genre_mask]
    filtered_scaled = scaled_df[genre_mask]
    
    # fit KNN
    knn = NearestNeighbors(n_neighbors=50, metric='cosine')
    knn.fit(filtered_scaled[features].values)
    distances, indices = knn.kneighbors(playlist_features)
    actual_indices = filtered_df.index[indices[0]]
    
    # exclude playlist songs and deduplicate by track_name
    seen_names = set()
    recommended = []
    for idx, dist in zip(actual_indices, distances[0]):
        if idx not in playlist_indices:
            name = df.loc[idx, 'track_name'].lower()
            if name not in seen_names:
                seen_names.add(name)
                recommended.append((idx, dist))
        if len(recommended) >= n:
            break
    
    rec_indices = [r[0] for r in recommended]
    rec_distances = [r[1] for r in recommended]
    
    results = df.loc[rec_indices][['track_name', 'artists', 'track_genre', 'popularity']].copy()
    results['distance'] = rec_distances
    results = results.reset_index(drop=True)
    results.index += 1
    
    return results

print('KNN v4 - Weighted + Multiple Genre + Deduplication\n')
results = recommend_knn_v4(playlist, df, full_features, df_full_standard)
print(results)

KNN v4 - Weighted + Multiple Genre + Deduplication

Playlist songs found: 5/5
Playlist:
  - Shape of You by Ed Sheeran (popularity: 86)
  - Blinding Lights by The Weeknd (popularity: 91)
  - Dance Monkey by Refeci;Michel Fannoun (popularity: 53)
  - Watermelon Sugar by Harry Styles (popularity: 89)
  - Levitating by Dua Lipa (popularity: 84)

Playlist genres: {'pop', 'electronic', 'dance', 'deep-house'}

                  track_name                   artists track_genre  \
1           Take You Dancing              Jason Derulo       dance   
2           Sweet but Psycho                   Ava Max       dance   
3   What Makes You Beautiful             One Direction         pop   
4                   Who Says  Selena Gomez & The Scene  dance, pop   
5                   Dynamite                       BTS  k-pop, pop   
6            Don't Start Now                  Dua Lipa       dance   
7                  New Rules                  Dua Lipa       dance   
8                Blank Space    

# KNN v4 Verification - Different Playlists

In [12]:
playlists = {
    'rock playlist': [
        'Bohemian Rhapsody',
        'Hotel California',
        'Stairway to Heaven',
        'Sweet Child O Mine',
    ],
    'sad playlist': [
        'Someone Like You',
        'The Night We Met',
        'Skinny Love',
        'Liability',
    ]
}

for name, songs in playlists.items():
    print(f'\n{"="*50}')
    print(f'Testing: {name}')
    print('='*50)
    results = recommend_knn_v4(songs, df, full_features, df_full_standard)
    if results is not None:
        print(results)


Testing: rock playlist
Songs not found: ['Hotel California', 'Stairway to Heaven', 'Sweet Child O Mine']
Playlist songs found: 1/4
Playlist:
  - Bohemian Rhapsody by Queen (popularity: 75)

Playlist genres: {'rock'}

                                           track_name  \
1                                           My Demons   
2                                    Comfortably Numb   
3                                Nothing Else Matters   
4                                          All I Want   
5                                           I See Red   
6                                          Body Paint   
7                                            Dream On   
8   A Thousand Years (feat. Steve Kazee) - Pt. 2; ...   
9                                              Zombie   
10                      Stairway to Heaven - Remaster   

                        artists             track_genre  popularity  distance  
1                       STARSET                    rock          73  0.209

# Save Best Model

In [13]:
import pickle

best_config = {
    'model': 'knn_v4',
    'features': 'full_features',
    'scaling': 'StandardScaler',
    'metric': 'cosine',
    'genre_filter': 'multiple_genre',
    'weighting': 'popularity_weighted',
    'deduplication': True,
    'use_most_popular': True
}

feature_matrix = df_full_standard[full_features].values
os.makedirs('../models/content_based', exist_ok=True)

np.save('../models/content_based/knn_feature_matrix.npy', feature_matrix)

with open('../models/content_based/knn_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

df[['track_id', 'track_name', 'artists', 'track_genre', 'popularity']].to_csv(
    '../models/content_based/knn_song_index.csv', index=True)

print('saved files:')
print('1. models/content_based/knn_feature_matrix.npy')
print('2. models/content_based/knn_config.json')
print('3. models/content_based/knn_song_index.csv')

saved files:
1. models/content_based/knn_feature_matrix.npy
2. models/content_based/knn_config.json
3. models/content_based/knn_song_index.csv


# Conclusion

In [14]:
print('='*50)
print('KNN MODEL - CONCLUSION')
print('='*50)

print('\n--- Experiments Summary ---')
print('Exp 1: full + standard + euclidean (no filter)     → failed: wrong genres')
print('Exp 2: full + standard + euclidean (genre filter)  → good but limited')
print('Exp 3: euclidean vs manhattan vs cosine            → cosine best metric')
print('Exp 4: audio vs full vs minmax                     → full + standard best')
print('Exp 5: weighted avg + multiple genre (v3)          → better results')
print('Exp 6: weighted + multiple genre + dedup (v4)      → best results')

print('\n--- Best Configuration (v4) ---')
print('Features     → full_features (21)')
print('Scaling      → StandardScaler')
print('Metric       → cosine')
print('Genre Filter → multiple genre (all playlist genres)')
print('Weighting    → popularity weighted average')
print('Dedup        → by track name')
print('Song Match   → most popular version')

print('\n--- Key Findings ---')
print('1. genre filter essential for relevant recommendations')
print('2. cosine metric outperforms euclidean and manhattan')
print('3. popularity weighted average better than simple average')
print('4. multiple genre filter better than dominant genre only')
print('5. deduplication needed to avoid same song appearing twice')
print('6. works well across different playlist types (rock, sad, pop)')
print('7. handles missing songs gracefully')

print('\n--- Limitations ---')
print('1. cover songs still affect playlist average')
print('2. short playlists (1 song) give limited genre pool')
print('3. very niche genres may have small recommendation pool')

print('\n--- Best Use Case ---')
print('playlist based recommendation')
print('works well for: recommend songs that fit my playlist vibe')

print('\n--- Saved Files ---')
print('models/content_based/knn_feature_matrix.npy')
print('models/content_based/knn_config.json')
print('models/content_based/knn_song_index.csv')

KNN MODEL - CONCLUSION

--- Experiments Summary ---
Exp 1: full + standard + euclidean (no filter)     → failed: wrong genres
Exp 2: full + standard + euclidean (genre filter)  → good but limited
Exp 3: euclidean vs manhattan vs cosine            → cosine best metric
Exp 4: audio vs full vs minmax                     → full + standard best
Exp 5: weighted avg + multiple genre (v3)          → better results
Exp 6: weighted + multiple genre + dedup (v4)      → best results

--- Best Configuration (v4) ---
Features     → full_features (21)
Scaling      → StandardScaler
Metric       → cosine
Genre Filter → multiple genre (all playlist genres)
Weighting    → popularity weighted average
Dedup        → by track name
Song Match   → most popular version

--- Key Findings ---
1. genre filter essential for relevant recommendations
2. cosine metric outperforms euclidean and manhattan
3. popularity weighted average better than simple average
4. multiple genre filter better than dominant genre only
